# 13: Time-Travel Queries for Historical Graph Analysis

This notebook demonstrates time-travel queries - the ability to query the graph state at any historical point in time.

## Key Features
- **Snapshot Queries**: "What did the graph look like on date X?"
- **Evolution Tracking**: "How did this network evolve over time?"
- **Audit Reconstruction**: "When was this suspicious pattern created?"
- **Temporal Comparison**: "What changed between date A and date B?"

## Use Cases
- Regulatory audits requiring historical snapshots
- Fraud investigation timeline reconstruction
- Compliance reporting with temporal context
- Pattern evolution analysis

## Time Periods in Demo
| Period | Date Range | Description |
|--------|------------|-------------|
| Initial | Jan 1-7, 2026 | Initial graph state |
| Growth | Jan 8-14, 2026 | Network expansion |
| Peak | Jan 15-21, 2026 | Peak activity |
| Suspicious | Jan 22-28, 2026 | Suspicious pattern emergence |
| Current | Jan 29 - Feb 4, 2026 | Current state |

**Author**: David LECONTE - IBM Worldwide | Data & AI | Tiger Team  
**Date**: 2026-03-23

In [ ]:
# Import required libraries
import sys
from pathlib import Path
from datetime import datetime, timedelta

# Add project root to path
project_root = Path().resolve().parent.parent
sys.path.insert(0, str(project_root))

# Fix asyncio event loop conflict in Jupyter
import nest_asyncio
nest_asyncio.apply()


import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import networkx as nx
from collections import defaultdict

from src.python.client.janusgraph_client import JanusGraphClient
from banking.analytics.time_travel import (
    TimeTravelQuery,
    TimeGranularity,
    create_time_travel_report
)

print("✅ Libraries imported successfully")


## 1. Connect to JanusGraph

In [ ]:
# Connect to JanusGraph using auto-detected config
from notebook_config import JANUSGRAPH_CONFIG
from urllib.parse import urlparse

# Parse the URL to get host/port
parsed = urlparse(JANUSGRAPH_CONFIG['url'])
host = parsed.hostname
port = parsed.port

client = JanusGraphClient(
    host=host,
    port=port,
    use_ssl=JANUSGRAPH_CONFIG['use_ssl'],
    verify_certs=False
)
client.connect()

# Verify connection
vertex_count = client.execute("g.V().count()")[0]
edge_count = client.execute("g.E().count()")[0]

print(f"Connected to JanusGraph at {host}:{port}")
print(f"Total vertices: {vertex_count}")
print(f"Total edges: {edge_count}")


## 2. Initialize Time-Travel Query Engine

In [ ]:
# Initialize time-travel query engine
tt_query = TimeTravelQuery(
    client=client,
    timestamp_property="createdAt",
    valid_from_property="validFrom",
    valid_to_property="validTo"
)

print("✅ Time-travel query engine initialized")

## 3. Generate Time-Travel Demo Data

In [ ]:
# Generate and load time-travel data
from banking.data_generators.scripts.generate_time_travel_data import (
    generate_time_travel_data_deterministic,
    load_into_janusgraph
)

# Generate data
print("Generating temporal graph data...")
data = generate_time_travel_data_deterministic(seed=42)

print(f"\nGenerated:")
print(f"  Vertices: {data['checksums']['total_vertices']}")
print(f"  Edges: {data['checksums']['total_edges']}")
print(f"  Persons: {data['checksums']['total_persons']}")
print(f"  Accounts: {data['checksums']['total_accounts']}")
print(f"  Transactions: {data['checksums']['total_transactions']}")

In [ ]:
# Load into JanusGraph
print("Loading into JanusGraph...")
counts = load_into_janusgraph(data)
print(f"Created {counts['vertices']} vertices and {counts['edges']} edges")

## 4. Snapshot Query: What did the graph look like on a specific date?

In [ ]:
# Query graph state at different points in time
dates = [
    datetime(2026, 1, 5, 12, 0, 0),   # Initial period
    datetime(2026, 1, 12, 12, 0, 0),  # Growth period
    datetime(2026, 1, 18, 12, 0, 0),  # Peak period
    datetime(2026, 1, 25, 12, 0, 0),  # Suspicious period
    datetime(2026, 2, 1, 12, 0, 0),   # Current
]

snapshots = []
for dt in dates:
    snapshot = tt_query.get_snapshot(dt)
    snapshots.append({
        "date": dt.strftime("%Y-%m-%d"),
        "vertices": snapshot.vertex_count,
        "edges": snapshot.edge_count
    })
    print(f"{dt.strftime('%Y-%m-%d')}: {snapshot.vertex_count} vertices, {snapshot.edge_count} edges")

In [ ]:
# Visualize graph evolution
df = pd.DataFrame(snapshots)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Vertex evolution
axes[0].bar(df['date'], df['vertices'], color='steelblue')
axes[0].set_title('Vertex Count Over Time', fontsize=14)
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Vertices')
axes[0].tick_params(axis='x', rotation=45)

# Edge evolution
axes[1].bar(df['date'], df['edges'], color='coral')
axes[1].set_title('Edge Count Over Time', fontsize=14)
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Edges')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('time_travel_evolution.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n📊 Graph evolution visualization saved")

## 5. Compare Snapshots: What changed between two dates?

In [ ]:
# Compare graph state at two different points
date_early = datetime(2026, 1, 10, 12, 0, 0)
date_late = datetime(2026, 1, 25, 12, 0, 0)

comparison = tt_query.compare_snapshots(date_early, date_late)

print(f"Comparison: {date_early.strftime('%Y-%m-%d')} vs {date_late.strftime('%Y-%m-%d')}")
print(f"\nChanges:")
print(f"  Vertices added: {comparison['vertices_added']}")
print(f"  Vertices removed: {comparison['vertices_removed']}")
print(f"  Vertices unchanged: {comparison['vertices_unchanged']}")
print(f"  Net vertex change: {comparison['vertex_count_change']}")
print(f"  Net edge change: {comparison['edge_count_change']}")

## 6. Timeline Analysis: Graph Evolution Over Time

In [ ]:
# Get detailed timeline over investigation period
start = datetime(2026, 1, 1, 0, 0, 0)
end = datetime(2026, 2, 4, 23, 59, 59)

timeline = tt_query.get_timeline(start, end, TimeGranularity.DAY)

# Convert to DataFrame for analysis
timeline_df = pd.DataFrame(timeline)
timeline_df['timestamp'] = pd.to_datetime(timeline_df['timestamp'])

print(f"Timeline: {len(timeline)} days of data")
print(timeline_df.head(10))

In [ ]:
# Visualize timeline
fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(timeline_df['timestamp'], timeline_df['vertex_count'], 'b-', label='Vertices', linewidth=2)
ax.plot(timeline_df['timestamp'], timeline_df['edge_count'], 'r-', label='Edges', linewidth=2)

# Highlight suspicious period
suspicious_start = datetime(2026, 1, 22)
suspicious_end = datetime(2026, 1, 28)
ax.axvspan(suspicious_start, suspicious_end, alpha=0.3, color='red', label='Suspicious Period')

ax.set_title('Graph Evolution Timeline', fontsize=14)
ax.set_xlabel('Date')
ax.set_ylabel('Count')
ax.legend()
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
plt.xticks(rotation=45)

plt.tight_layout()
plt.savefig('time_travel_timeline.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n📊 Timeline visualization saved")

## 7. Generate Investigation Report

In [ ]:
# Generate comprehensive time-travel investigation report
investigation_date = datetime(2026, 2, 1, 12, 0, 0)
report = create_time_travel_report(
    client=client,
    investigation_date=investigation_date,
    investigation_period_days=30
)

print("=" * 60)
print("TIME-TRAVEL INVESTIGATION REPORT")
print("=" * 60)
print(f"\nInvestigation Date: {report['investigation_date']}")
print(f"Period Start: {report['period_start']}")
print(f"Period Days: {report['period_days']}")
print(f"\nSummary:")
print(f"  Vertices at start: {report['summary']['vertices_at_start']}")
print(f"  Vertices at end: {report['summary']['vertices_at_end']}")
print(f"  Net change: {report['summary']['net_change']}")
print(f"  Edges at start: {report['summary']['edges_at_start']}")
print(f"  Edges at end: {report['summary']['edges_at_end']}")

## 8. Regulatory Compliance Context

Time-travel queries support the following regulatory requirements:

### GDPR Article 30 - Records of Processing Activities
Ability to reconstruct data state at any historical point.

### FinCEN SAR Reporting
Timeline reconstruction for suspicious activity reports.

### BSA/AML Record Keeping
5-7 year retention with queryable historical state.

### Audit Trail Requirements
Complete provenance and change history for regulators.

In [ ]:
# Save report for compliance
import json

report_path = Path("exports/time-travel/investigation_report.json")
report_path.parent.mkdir(parents=True, exist_ok=True)

with open(report_path, 'w') as f:
    json.dump(report, f, indent=2, default=str)

print(f"✅ Report saved to: {report_path}")

## Summary

This notebook demonstrated Time-Travel Queries for historical graph analysis:

1. **Snapshot Queries** - Query graph state at any historical point
2. **Temporal Comparison** - Compare graph states between dates
3. **Timeline Analysis** - Track graph evolution over time
4. **Investigation Reports** - Generate compliance-ready reports

### Key Benefits
- Regulatory compliance (GDPR, BSA/AML, FinCEN)
- Audit trail reconstruction
- Fraud investigation timeline
- Pattern evolution analysis

---
**Author**: David LECONTE - IBM Worldwide | Data & AI | Tiger Team  